<a href="https://www.kaggle.com/code/abubacker/customer-churn-prediction?scriptVersionId=189181623" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import warnings
warnings.filterwarnings('ignore')
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
from tensorflow import keras
import seaborn as sns 
import matplotlib.pyplot as plt

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### Data

In [ ]:
data = pd.read_csv(r'/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
data.head()

### EDA

In [ ]:
data.describe()

In [ ]:
data.info()

In [ ]:
df = data.drop(['customerID'],axis=1)

In [ ]:
df.TotalCharges.dtype

In [ ]:
# pd.to_numeric(df.TotalCharges) - 
# ValueError: Unable to parse string " " at position 488

In [ ]:
df[df.TotalCharges==' ']

In [ ]:
df = df[df.TotalCharges!=' ']

In [ ]:
df.TotalCharges = pd.to_numeric(df.TotalCharges)

In [ ]:
sns.countplot(data=df,x='Churn')

In [ ]:
sns.histplot(data=data,x='MonthlyCharges',hue='Churn',kde=True)

In [ ]:
sns.histplot(x=df.TotalCharges,hue=df.Churn,kde=True)

In [ ]:
sns.histplot(x=df.tenure,hue=df.Churn,kde=True)

In [ ]:
sns.countplot(data=df,x='gender',hue='Churn')

In [ ]:
df.columns

In [ ]:
pd.crosstab(df.Contract,df.Churn).plot()

In [ ]:
pd.crosstab(df.Contract,df.Churn).plot()

In [ ]:
sns.heatmap(pd.crosstab(df.Contract,df.Churn),annot=True,fmt='g')

In [ ]:
pd.crosstab(df.Churn,df.PaymentMethod)

### Preprocessing

In [ ]:
#we found that many cells has No internet service instead of NO

In [ ]:
df.replace('No internet service','No',inplace=True)

In [ ]:
df.MultipleLines.value_counts()
df.replace('No phone service','No',inplace=True)

In [ ]:
for i in df.columns:
    print(i,':',df[i].unique())

In [ ]:
df.gender = df.gender.replace({'Male':1,'Female':0})

In [ ]:
cat_cols = ['Partner','Dependents','PhoneService','MultipleLines','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','PaperlessBilling','Churn']

In [ ]:
df[cat_cols] = df[cat_cols].replace({'Yes':1,'No':0})

In [ ]:
df = pd.get_dummies(df,drop_first=True,dtype='uint')

In [ ]:
num_cols = ['tenure','MonthlyCharges','TotalCharges']

In [ ]:
mn = MinMaxScaler()
df[num_cols] = mn.fit_transform(df[num_cols])

In [ ]:
df.sample(3)

### Modelling

In [ ]:
X = df.drop('Churn',axis=1)
y = df.Churn

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2)

In [ ]:
model = keras.Sequential()

In [ ]:
X_train.shape

In [ ]:
model.add(
keras.layers.Dense(30,input_shape=(23,),activation='relu'))
model.add(
keras.layers.Dense(10,activation='relu'))
model.add(
keras.layers.Dense(1,activation='sigmoid')
)

In [ ]:
model.summary()

In [ ]:
callbacks = keras.callbacks.EarlyStopping(min_delta=0.001,patience = 3)


In [ ]:
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=[ keras.metrics.BinaryAccuracy()])

In [ ]:
model.fit(x=X_train,y=y_train,batch_size=120,epochs=60,callbacks=[callbacks],validation_data=(X_test,y_test))

In [ ]:
history = pd.DataFrame(model.history.history)

In [ ]:
history[['binary_accuracy','val_binary_accuracy']].plot()

In [ ]:
history[['loss','val_loss']].plot()

### Inferencing..

In [ ]:
model.layers

In [ ]:
model.weights

In [ ]:
# help(model)
y_pred = np.where(model.predict(X_test)>0.5,1,0).reshape(-1)

In [ ]:
accuracy_score(y_test,y_pred)

In [ ]:
print(classification_report(y_test,y_pred)) #since the dist of data along class 0 and 1 is not normal ( some errs occurs...!)

In [ ]:
sns.heatmap(confusion_matrix(y_test,y_pred),annot=True,fmt='g',cmap='Blues')

### Thats'all END! 

In [ ]:
#Abubacker S